1 epoch

In [1]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-70B-Instruct-Reference-GO-CC-70B-Term-ID-10-3a8cef1d",  # Your fine-tuned model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in Gene Ontology (GO). Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since GO IDs are short and your model is trained for this specific task
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"GO_ID": "Error"})

# Improved function to extract JSON from the LLM response for GO data
def extract_go_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'GO_ID' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'GO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"GO_ID"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'GO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract GO_ID value directly
        go_id_match = re.search(r'"GO_ID"[\s:]*"([^"]+)"', response_text)
        if go_id_match:
            return {"GO_ID": go_id_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        go_id_match = re.search(r'GO_ID["\':\s]+([^"\'}\s,]+)', response_text)
        if go_id_match:
            return {"GO_ID": go_id_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"GO_ID": "None"}

        # If all parsing attempts fail
        return {"GO_ID": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"GO_ID": "Parse_Error"}

# Enhanced prompt to get GO ID for a given GO term
def get_go_id_from_term(go_term):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the Gene Ontology term "{go_term}", provide the corresponding GO ID.

Instructions:
- Provide the EXACT official GO ID as it appears in the Gene Ontology database
- Choose the most direct, standard GO ID for the term
- Avoid overly specific or complex variations (e.g., prefer primary GO terms over highly specific subterms)
- Return the primary, commonly used GO ID
- GO IDs follow the format: GO:XXXXXXX (where X are 7 digits)
- Examples: GO:0008150, GO:0003674, GO:0005575
- If the term is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "GO_ID": "<go_id_here>"
}}

GO Term: {go_term}"""

    response_text = query_together(prompt)
    result = extract_go_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "go_term": go_term,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("finetuned70B_go_api_responses.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("GO_ID", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_go_id, new_go_id):
    """
    Calculate match result between original and new GO IDs
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_go_id) or pd.isna(new_go_id):
        return 0
    return int(str(original_go_id).strip() == str(new_go_id).strip())

# Main processing function
def main():
    print("Starting GO term mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("finetuned70B_go_api_responses.jsonl"):
        with open("finetuned70B_go_api_responses.jsonl", "w") as f:
            pass

    # Test the API with a sample GO term
    print("Testing API connection...")
    test_result = get_go_id_from_term("apoptotic process")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the GO terms CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\go_terms.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["newgoid"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            go_term = row["go_term"]  # Using the go_term column from the CSV
            original_go_id = row["go_id"]  # Original GO ID for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {go_term}")

            start_time = time.time()
            new_go_id = get_go_id_from_term(go_term)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new GO ID
            df.loc[idx, "newgoid"] = new_go_id
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_go_id, new_go_id)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_go_id}, New: {new_go_id}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_term', 'normalized_id', 'go_match']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned70B_go_term_id_results_progress10.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} terms.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "newgoid"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_term', 'normalized_id', 'go_match']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned70B_go_term_id_results_final10.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model) ===")
        print(f"Total terms processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["newgoid"] == "None").sum()
        error_results = (df["newgoid"] == "Error").sum()
        parse_error_results = (df["newgoid"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid GO IDs returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["newgoid"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about GO aspects if available
        if 'Aspect' in df.columns:
            print(f"\nBreakdown by GO Aspect:")
            aspect_stats = df.groupby('Aspect').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(aspect_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned70B_go_term_id_results_final10.csv")
    print(f"Progress file: finetuned70B_go_term_id_results_progress10.csv")
    print(f"Log file: finetuned70B_go_api_responses10.jsonl")
    print(f"Columns excluded from output: normalized_term, normalized_id, go_match")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting GO term mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: GO:0006915
API test successful, proceeding with batch processing...
Loaded 1839 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\go_terms.csv
Columns in the dataset: ['go_id', 'go_annotations', 'go_term', 'Definitions', 'Parents', 'Aspect', 'go_id_pmc', 'go_term_pmc', 'normalized_term', 'normalized_id', 'go_match']
Processing 1/1839: cytosol
Sending request (attempt 1/5)...
Error processing row 0: [Errno 22] Invalid argument
Processing 2/1839: nucleoplasm
Sending request (attempt 1/5)...
Error processing row 1: [Errno 22] Invalid argument
Processing 3/1839: plasma membrane
Sending request (attempt 1/5)...
Error processing row 2: [Errno 22] Invalid argument
Processing 4/1839: extracellular region
Sending request (attempt 1/5)...
Error processing row 3: [Errno 22] Invalid argument
Processing 5/1839: 

In [2]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-70B-Instruct-Reference-GO-CC-70B-Term-ID-20-7dc1df81",  # Your fine-tuned model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in Gene Ontology (GO). Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since GO IDs are short and your model is trained for this specific task
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"GO_ID": "Error"})

# Improved function to extract JSON from the LLM response for GO data
def extract_go_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'GO_ID' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'GO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"GO_ID"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'GO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract GO_ID value directly
        go_id_match = re.search(r'"GO_ID"[\s:]*"([^"]+)"', response_text)
        if go_id_match:
            return {"GO_ID": go_id_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        go_id_match = re.search(r'GO_ID["\':\s]+([^"\'}\s,]+)', response_text)
        if go_id_match:
            return {"GO_ID": go_id_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"GO_ID": "None"}

        # If all parsing attempts fail
        return {"GO_ID": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"GO_ID": "Parse_Error"}

# Enhanced prompt to get GO ID for a given GO term
def get_go_id_from_term(go_term):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the Gene Ontology term "{go_term}", provide the corresponding GO ID.

Instructions:
- Provide the EXACT official GO ID as it appears in the Gene Ontology database
- Choose the most direct, standard GO ID for the term
- Avoid overly specific or complex variations (e.g., prefer primary GO terms over highly specific subterms)
- Return the primary, commonly used GO ID
- GO IDs follow the format: GO:XXXXXXX (where X are 7 digits)
- Examples: GO:0008150, GO:0003674, GO:0005575
- If the term is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "GO_ID": "<go_id_here>"
}}

GO Term: {go_term}"""

    response_text = query_together(prompt)
    result = extract_go_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "go_term": go_term,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("finetuned70B_go_api_responses.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("GO_ID", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_go_id, new_go_id):
    """
    Calculate match result between original and new GO IDs
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_go_id) or pd.isna(new_go_id):
        return 0
    return int(str(original_go_id).strip() == str(new_go_id).strip())

# Main processing function
def main():
    print("Starting GO term mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("finetuned70B_go_api_responses.jsonl"):
        with open("finetuned70B_go_api_responses.jsonl", "w") as f:
            pass

    # Test the API with a sample GO term
    print("Testing API connection...")
    test_result = get_go_id_from_term("apoptotic process")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the GO terms CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\go_terms.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["newgoid"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            go_term = row["go_term"]  # Using the go_term column from the CSV
            original_go_id = row["go_id"]  # Original GO ID for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {go_term}")

            start_time = time.time()
            new_go_id = get_go_id_from_term(go_term)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new GO ID
            df.loc[idx, "newgoid"] = new_go_id
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_go_id, new_go_id)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_go_id}, New: {new_go_id}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_term', 'normalized_id', 'go_match']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned70B_go_term_id_results_progress20.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} terms.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "newgoid"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_term', 'normalized_id', 'go_match']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned70B_go_term_id_results_final20.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model) ===")
        print(f"Total terms processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["newgoid"] == "None").sum()
        error_results = (df["newgoid"] == "Error").sum()
        parse_error_results = (df["newgoid"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid GO IDs returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["newgoid"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about GO aspects if available
        if 'Aspect' in df.columns:
            print(f"\nBreakdown by GO Aspect:")
            aspect_stats = df.groupby('Aspect').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(aspect_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned70B_go_term_id_results_final20.csv")
    print(f"Progress file: finetuned70B_go_term_id_results_progress20.csv")
    print(f"Log file: finetuned70B_go_api_responses20.jsonl")
    print(f"Columns excluded from output: normalized_term, normalized_id, go_match")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting GO term mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: GO:0006915
API test successful, proceeding with batch processing...
Loaded 1839 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\go_terms.csv
Columns in the dataset: ['go_id', 'go_annotations', 'go_term', 'Definitions', 'Parents', 'Aspect', 'go_id_pmc', 'go_term_pmc', 'normalized_term', 'normalized_id', 'go_match']
Processing 1/1839: cytosol
Sending request (attempt 1/5)...
  Original: GO:0005829, New: GO:0005829, Match: 1
Progress saved. Processed 1/1839 terms.
Waiting 2.79 seconds before next request...
Processing 2/1839: nucleoplasm
Sending request (attempt 1/5)...
  Original: GO:0005654, New: GO:0005654, Match: 1
Waiting 2.56 seconds before next request...
Processing 3/1839: plasma membrane
Sending request (attempt 1/5)...
  Original: GO:0005886, New: GO:0044298, Match: 0
Waiting 2.86 seconds b

In [3]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-70B-Instruct-Reference-GO-CC-70B-Term-ID-15-62eac8f2",  # Your fine-tuned model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in Gene Ontology (GO). Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since GO IDs are short and your model is trained for this specific task
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"GO_ID": "Error"})

# Improved function to extract JSON from the LLM response for GO data
def extract_go_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'GO_ID' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'GO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"GO_ID"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'GO_ID' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract GO_ID value directly
        go_id_match = re.search(r'"GO_ID"[\s:]*"([^"]+)"', response_text)
        if go_id_match:
            return {"GO_ID": go_id_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        go_id_match = re.search(r'GO_ID["\':\s]+([^"\'}\s,]+)', response_text)
        if go_id_match:
            return {"GO_ID": go_id_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"GO_ID": "None"}

        # If all parsing attempts fail
        return {"GO_ID": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"GO_ID": "Parse_Error"}

# Enhanced prompt to get GO ID for a given GO term
def get_go_id_from_term(go_term):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the Gene Ontology term "{go_term}", provide the corresponding GO ID.

Instructions:
- Provide the EXACT official GO ID as it appears in the Gene Ontology database
- Choose the most direct, standard GO ID for the term
- Avoid overly specific or complex variations (e.g., prefer primary GO terms over highly specific subterms)
- Return the primary, commonly used GO ID
- GO IDs follow the format: GO:XXXXXXX (where X are 7 digits)
- Examples: GO:0008150, GO:0003674, GO:0005575
- If the term is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "GO_ID": "<go_id_here>"
}}

GO Term: {go_term}"""

    response_text = query_together(prompt)
    result = extract_go_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "go_term": go_term,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("finetuned70B_go_api_responses.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("GO_ID", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_go_id, new_go_id):
    """
    Calculate match result between original and new GO IDs
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_go_id) or pd.isna(new_go_id):
        return 0
    return int(str(original_go_id).strip() == str(new_go_id).strip())

# Main processing function
def main():
    print("Starting GO term mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("finetuned70B_go_api_responses.jsonl"):
        with open("finetuned70B_go_api_responses.jsonl", "w") as f:
            pass

    # Test the API with a sample GO term
    print("Testing API connection...")
    test_result = get_go_id_from_term("apoptotic process")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the GO terms CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\go_terms.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["newgoid"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            go_term = row["go_term"]  # Using the go_term column from the CSV
            original_go_id = row["go_id"]  # Original GO ID for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {go_term}")

            start_time = time.time()
            new_go_id = get_go_id_from_term(go_term)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new GO ID
            df.loc[idx, "newgoid"] = new_go_id
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_go_id, new_go_id)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_go_id}, New: {new_go_id}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_term', 'normalized_id', 'go_match']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned70B_go_term_id_results_progress15.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} terms.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "newgoid"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_term', 'normalized_id', 'go_match']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned70B_go_term_id_results_final15.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model) ===")
        print(f"Total terms processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["newgoid"] == "None").sum()
        error_results = (df["newgoid"] == "Error").sum()
        parse_error_results = (df["newgoid"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid GO IDs returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["newgoid"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about GO aspects if available
        if 'Aspect' in df.columns:
            print(f"\nBreakdown by GO Aspect:")
            aspect_stats = df.groupby('Aspect').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(aspect_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned70B_go_term_id_results_final15.csv")
    print(f"Progress file: finetuned70B_go_term_id_results_progress15.csv")
    print(f"Log file: finetuned70B_go_api_responses15.jsonl")
    print(f"Columns excluded from output: normalized_term, normalized_id, go_match")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting GO term mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: GO:0006915
API test successful, proceeding with batch processing...
Loaded 1839 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\go_terms.csv
Columns in the dataset: ['go_id', 'go_annotations', 'go_term', 'Definitions', 'Parents', 'Aspect', 'go_id_pmc', 'go_term_pmc', 'normalized_term', 'normalized_id', 'go_match']
Processing 1/1839: cytosol
Sending request (attempt 1/5)...
  Original: GO:0005829, New: GO:0005829, Match: 1
Progress saved. Processed 1/1839 terms.
Waiting 2.54 seconds before next request...
Processing 2/1839: nucleoplasm
Sending request (attempt 1/5)...
  Original: GO:0005654, New: GO:0005654, Match: 1
Waiting 3.56 seconds before next request...
Processing 3/1839: plasma membrane
Sending request (attempt 1/5)...
  Original: GO:0005886, New: GO:0044298, Match: 0
Waiting 2.21 seconds b